# Setup the Environment

In [1]:
!nvidia-smi
!pip install -q transformers datasets accelerate scikit-learn

Tue Nov 10 02:04:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Connected to Google Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# Load the Data

In [3]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

DATA = "/content/drive/MyDrive/fake-news/processed_v3"
OUT = "/content/drive/MyDrive/fake-news/models/distilroberta_v3"
MODEL_NAME = "distilroberta-base"
MAX_LEN = 256   # only the first 256 tokens of each article are used

print("GPU available:", torch.cuda.is_available())


def load(split):
    df = pd.read_csv(f"{DATA}/{split}.csv")
    df["title"] = df["title"].fillna("")
    df["text"] = df["text"].fillna("")
    df["full"] = df["title"] + ". " + df["text"]
    return df


# Label meaning: 0 = real, 1 = fake.
# The test set is deliberately NOT loaded here.
train_df, val_df = load("train"), load("val")
print(len(train_df), len(val_df))

GPU available: True
48918 6115


# Tokenize

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(df):
    ds = Dataset.from_dict({"text": df["full"].tolist(), "labels": df["label"].tolist()})
    return ds.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LEN),
        batched=True,
        remove_columns=["text"],
    )

train_ds, val_ds = to_dataset(train_df), to_dataset(val_df)

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/48918 [00:00<?, ? examples/s]

Map:   0%|          | 0/6115 [00:00<?, ? examples/s]

# Train original model

*   WARNING: Training takes about 12 minutes and OVERWRITES the saved model in Drive.
*   Do NOT run this cell (or the next one) again. Use "Load saved model" instead.




In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "real", 1: "fake"},
    label2id={"real": 0, "fake": 1},
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


# 48,918 articles / batch size 16 = about 3,057 steps per epoch, so 2 epochs = about 6,100 steps.
# warmup_steps=600 is about 10% of that: the learning rate rises slowly for the first 600 steps.
args = TrainingArguments(
    output_dir="/content/checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=600,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    fp16=True,
    logging_steps=200,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.020927,0.033734,0.992968,0.992847
2,0.004201,0.022982,0.995912,0.995845


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6116, training_loss=0.045272178523988016, metrics={'train_runtime': 726.3093, 'train_samples_per_second': 134.703, 'train_steps_per_second': 8.421, 'total_flos': 6480040207454208.0, 'train_loss': 0.045272178523988016, 'epoch': 2.0})

# Evaluate and save

WARNING: This cell saves to OUT and OVERWRITES the saved model. Do not run again.

In [ ]:
pred = trainer.predict(val_ds)
y_pred = np.argmax(pred.predictions, axis=-1)
y_true = pred.label_ids

print(classification_report(y_true, y_pred, target_names=["real", "fake"], digits=4))
print("Confusion matrix (rows = true, columns = predicted; order: real, fake):")
print(confusion_matrix(y_true, y_pred))

trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)
print("Saved to", OUT)

              precision    recall  f1-score   support

        real     0.9954    0.9974    0.9964      3441
        fake     0.9966    0.9940    0.9953      2674

    accuracy                         0.9959      6115
   macro avg     0.9960    0.9957    0.9958      6115
weighted avg     0.9959    0.9959    0.9959      6115

Confusion matrix (rows = true, columns = predicted; order: real, fake):
[[3432    9]
 [  16 2658]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/fake-news/models/distilroberta_v3


# Load saved model from Drive

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(OUT)
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="/content/tmp",
        per_device_eval_batch_size=64,
        fp16=True,
        report_to="none",
    ),
    data_collator=DataCollatorWithPadding(tokenizer),
)
print("Model loaded from Drive")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded from Drive


# Robustness check: style and formatting stress test

Re-evaluates the same validation articles after removing style and formatting cues (case, punctuation, title, first 300 characters) to see how much the model relies on them.

In [6]:
import re


def normalize(text):
    """Lowercase, plain quotes, no punctuation: removes style and formatting cues."""
    text = text.lower()
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def run_variant(name, full_series):
    df = val_df.copy()
    df["full"] = full_series
    ds = to_dataset(df)
    out = trainer.predict(ds)
    preds = np.argmax(out.predictions, axis=-1)
    acc = accuracy_score(df["label"], preds)
    f1 = f1_score(df["label"], preds, average="macro")
    print(f"{name:38s} accuracy={acc:.4f}  macro_f1={f1:.4f}")


run_variant("1. original", val_df["full"])
run_variant("2. lowercase, no punctuation", val_df["full"].apply(normalize))
run_variant("3. title only", val_df["title"])
run_variant("4. body only (no title)", val_df["text"])
run_variant("5. body without first 300 chars", val_df["text"].str[300:])

Map:   0%|          | 0/6115 [00:00<?, ? examples/s]

1. original                            accuracy=0.9959  macro_f1=0.9958


Map:   0%|          | 0/6115 [00:00<?, ? examples/s]

2. lowercase, no punctuation           accuracy=0.7452  macro_f1=0.7419


Map:   0%|          | 0/6115 [00:00<?, ? examples/s]

3. title only                          accuracy=0.9066  macro_f1=0.9063


Map:   0%|          | 0/6115 [00:00<?, ? examples/s]

4. body only (no title)                accuracy=0.9797  macro_f1=0.9793


Map:   0%|          | 0/6115 [00:00<?, ? examples/s]

5. body without first 300 chars        accuracy=0.9195  macro_f1=0.9178


# Content-only (normalized) experiment

In [7]:
OUT_NORM = "/content/drive/MyDrive/fake-news/models/distilroberta_v3_normalized"

# Normalize the text for training AND validation (same function as the robustness check)
train_norm = train_df.copy()
val_norm = val_df.copy()
train_norm["full"] = train_norm["full"].apply(normalize)
val_norm["full"] = val_norm["full"].apply(normalize)
train_ds_n, val_ds_n = to_dataset(train_norm), to_dataset(val_norm)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


model_n = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "real", 1: "fake"},
    label2id={"real": 0, "fake": 1},
)

args_n = TrainingArguments(
    output_dir="/content/checkpoints_norm",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=600,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    fp16=True,
    logging_steps=200,
    report_to="none",
    seed=42,
)

trainer_n = Trainer(
    model=model_n,
    args=args_n,
    train_dataset=train_ds_n,
    eval_dataset=val_ds_n,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer_n.train()

# Evaluate on the normalized validation set and save to Drive
pred = trainer_n.predict(val_ds_n)
y_pred = np.argmax(pred.predictions, axis=-1)
print(classification_report(pred.label_ids, y_pred, target_names=["real", "fake"], digits=4))
print(confusion_matrix(pred.label_ids, y_pred))

trainer_n.save_model(OUT_NORM)
tokenizer.save_pretrained(OUT_NORM)
print("Saved to", OUT_NORM)

Map:   0%|          | 0/48918 [00:00<?, ? examples/s]

Map:   0%|          | 0/6115 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

        real     0.9748    0.9762    0.9755      3441
        fake     0.9693    0.9675    0.9684      2674

    accuracy                         0.9724      6115
   macro avg     0.9720    0.9718    0.9719      6115
weighted avg     0.9724    0.9724    0.9724      6115

[[3359   82]
 [  87 2587]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/fake-news/models/distilroberta_v3_normalized
